Mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

Import in bike lanes, speeding, and night

In [ ]:
import pandas as pd
import geopandas as gpd

# Paths
speeding_path = "/content/drive/MyDrive/cluster_threshold_tables/cluster_59_4.csv"
night_path    = "/content/drive/MyDrive/night_threshold_tables/night_30_csv"
bike_path     = "/content/drive/MyDrive/bike_threshold_tables/bike_100_csv"

# Simple loader
def load_file(path):
    try:
        return pd.read_csv(path)
    except:
        return gpd.read_file(path)

# Load tables
speeding_df = load_file(speeding_path)
night_df    = load_file(night_path)
bike_df     = load_file(bike_path)

# Add source labels
speeding_df["source_type"] = "speeding"
night_df["source_type"]    = "night"
bike_df["source_type"]     = "bike"

# Preview
print("Speeding:", speeding_df.shape)
print("Night:", night_df.shape)
print("Bike:", bike_df.shape)

Standardize Format Across All Tables

In [ ]:
def standardize_hotspot_table(df):
    out = df.copy()

    # unify names
    rename_map = {
        "N_CRASHES": "n_crashes",
        "SEVERITY_SUM": "severity_sum",
        "CAPPED_SEVERITY_SUM": "capped_severity_sum",
        "AVG_SEVERITY": "avg_severity",
        "AVG_CAPPED_SEVERITY": "avg_capped_severity",
        "SEVERE_CRASH_PERCENTAGE": "severe_pct",
        "FATAL_CRASH_PERCENTAGE": "fatal_pct",
        "AVG_LAT": "lat",
        "AVG_LON": "lon"
    }

    out = out.rename(columns=rename_map)

    # keep only common useful columns
    keep_cols = [
        "source_type",
        "n_crashes",
        "severity_sum",
        "capped_severity_sum",
        "avg_severity",
        "avg_capped_severity",
        "severe_pct",
        "fatal_pct",
        "lat",
        "lon"
    ]

    out = out[keep_cols].copy()

    return out

Apply to all tables

In [ ]:
speeding_std = standardize_hotspot_table(speeding_df)
night_std    = standardize_hotspot_table(night_df)
bike_std     = standardize_hotspot_table(bike_df)

Import in crash data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.cluster.hierarchy import linkage, fcluster

# -------------------
# Paths
# -------------------
CRASHES_PATH = "/content/drive/MyDrive/Crashes_in_DC.csv"

# -------------------
# Parameters
# -------------------
DATE_START = "2020-01-01"
DATE_END   = "2025-04-30"
MAR_MIN    = 100
CRS_METERS = 32618

# Pick your clustering cutoff
THRESHOLD_M = 100

helper functions

In [ ]:
def clean_intersection(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip()
    if re.fullmatch(r"(nan|none|null|missing)", s, flags=re.I):
        return ""
    if "Intersecting RouteID" in s or "*" in s:
        return ""
    s = re.sub(r"\s*[/@&]\s*", " & ", s)
    return re.sub(r"\s+", " ", s).strip()

def extract_street_name(row):
    txt = None
    if "MAR_Address" in row and isinstance(row["MAR_Address"], str):
        txt = row["MAR_Address"]
    elif "ADDRESS" in row and isinstance(row["ADDRESS"], str):
        txt = row["ADDRESS"]

    if txt:
        t = txt.upper().strip()
        t = re.split(r"\s*&\s*|\s*@\s*|\s*/\s*|,| - ", t)[0]
        t = re.sub(r"\b(NE|NW|SE|SW)\b", "", t)
        return re.sub(r"\s+", " ", t).strip()

    inter = str(row.get("NEARESTINTSTREETNAME", "")).upper().strip()
    if inter:
        return re.split(r"\s*&\s*|\s*@\s*|\s*/\s*|,| - ", inter)[0].strip()

    return ""

def standardize_name(df, out_col="NAME", candidates=None, fallback_series=None):
    if candidates is None:
        candidates = []

    out = pd.Series("", index=df.index, dtype=object)

    for c in candidates:
        if c in df.columns:
            cand = df[c].fillna("").astype(str).str.strip()
            out = np.where(out != "", out, cand)

    if fallback_series is not None:
        fb = fallback_series.fillna("").astype(str).str.strip()
        out = np.where(out != "", out, fb)

    out = pd.Series(out, index=df.index).replace("", "(UNNAMED)")
    df[out_col] = out
    return df

Load and clean all crashes

In [ ]:
df = pd.read_csv(CRASHES_PATH, dtype={"STREETSEGID": str}, low_memory=False)

# Valid lat/lon
df = df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
df["LATITUDE"] = pd.to_numeric(df["LATITUDE"], errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")
df = df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

# Date window
df["FROMDATE"] = pd.to_datetime(df["FROMDATE"], errors="coerce")
df = df[(df["FROMDATE"] >= DATE_START) & (df["FROMDATE"] <= DATE_END)].copy()

# MAR quality
if "MAR_SCORE" in df.columns:
    df["MAR_SCORE"] = pd.to_numeric(df["MAR_SCORE"], errors="coerce")
    df = df[df["MAR_SCORE"] >= MAR_MIN].copy()

print(f"Crashes after basic filters: {len(df):,}")

create severity

In [ ]:
injury_categories = {
    "BICYCLIST":  ["MAJORINJURIES_BICYCLIST","MINORINJURIES_BICYCLIST","UNKNOWNINJURIES_BICYCLIST","FATAL_BICYCLIST"],
    "DRIVER":     ["MAJORINJURIES_DRIVER","MINORINJURIES_DRIVER","UNKNOWNINJURIES_DRIVER","FATAL_DRIVER"],
    "PEDESTRIAN": ["MAJORINJURIES_PEDESTRIAN","MINORINJURIES_PEDESTRIAN","UNKNOWNINJURIES_PEDESTRIAN","FATAL_PEDESTRIAN"],
    "PASSENGER":  ["MAJORINJURIESPASSENGER","MINORINJURIESPASSENGER","FATALPASSENGER"],
    "OTHER":      ["MAJORINJURIESOTHER","MINORINJURIESOTHER","FATALOTHER"],
}

fatal_cols = [c for cols in injury_categories.values() for c in cols if "FATAL" in c]
major_cols = [c for cols in injury_categories.values() for c in cols if "MAJOR" in c]
minor_cols = [c for cols in injury_categories.values() for c in cols if "MINOR" in c]

for col in set(fatal_cols + major_cols + minor_cols):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    else:
        df[col] = 0

injury_cols_all = fatal_cols + major_cols + minor_cols
df = df[df[injury_cols_all].sum(axis=1) > 0].copy()

df["TOTAL_FATALITIES"] = df[fatal_cols].sum(axis=1)
df["TOTAL_MAJOR_INJURIES"] = df[major_cols].sum(axis=1)
df["TOTAL_MINOR_INJURIES"] = df[minor_cols].sum(axis=1)

df["SEVERITY_SCORE"] = (
    7 * df["TOTAL_FATALITIES"] +
    4 * df["TOTAL_MAJOR_INJURIES"] +
    1 * df["TOTAL_MINOR_INJURIES"]
)

df["CAPPED_SEVERITY"] = np.where(
    df["TOTAL_FATALITIES"] > 0, 7,
    np.where(df["TOTAL_MAJOR_INJURIES"] > 0, 4,
             np.where(df["TOTAL_MINOR_INJURIES"] > 0, 1, 0))
)

df["IS_SEVERE_CRASH"] = (
    (df["TOTAL_FATALITIES"] > 0) | (df["TOTAL_MAJOR_INJURIES"] > 0)
).astype(int)

df["IS_FATAL_CRASH"] = (df["TOTAL_FATALITIES"] > 0).astype(int)

if "NEARESTINTSTREETNAME" not in df.columns:
    df["NEARESTINTSTREETNAME"] = ""

df["NEARESTINTSTREETNAME"] = df["NEARESTINTSTREETNAME"].apply(clean_intersection)
df["PRIMARY_STREET"] = df.apply(extract_street_name, axis=1).str.upper()

df = standardize_name(
    df,
    out_col="NAME",
    candidates=["PRIMARY_STREET", "NEARESTINTSTREETNAME"]
)

print(f"Crashes after injury/severity filters: {len(df):,}")
display(df[[
    "SEVERITY_SCORE",
    "CAPPED_SEVERITY",
    "IS_SEVERE_CRASH",
    "IS_FATAL_CRASH"
]])

Convert to meters for clustering

In [ ]:
gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["LONGITUDE"], df["LATITUDE"]),
    crs=4326
).to_crs(CRS_METERS)

gdf = gdf.reset_index(drop=True)
gdf["crash_index"] = gdf.index

print(gdf.shape)
display(gdf[["crash_index", "SEVERITY_SCORE", "CAPPED_SEVERITY", "geometry"]].head())

cluster with complete linkage

In [ ]:
coords = np.c_[gdf.geometry.x.values, gdf.geometry.y.values]

if len(coords) == 1:
    gdf["cluster_id"] = 0
else:
    Z = linkage(coords, method="complete", metric="euclidean")
    gdf["cluster_id"] = fcluster(Z, t=THRESHOLD_M, criterion="distance") - 1

print("Number of clusters:", gdf["cluster_id"].nunique())
display(gdf[["crash_index", "cluster_id", "SEVERITY_SCORE"]].head())

create tables

In [ ]:
gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

# Rebuild clean summary table from scratch
cluster_df = (
    gdf.groupby("cluster_id")
       .agg(
           N_CRASHES=("crash_index", "size"),
           SEVERITY_SUM=("SEVERITY_SCORE", "sum"),
           CAPPED_SEVERITY_SUM=("CAPPED_SEVERITY", "sum"),
           SEVERE_CRASH_COUNT=("IS_SEVERE_CRASH", "sum"),
           FATAL_CRASH_COUNT=("IS_FATAL_CRASH", "sum"),
           avg_x=("x", "mean"),
           avg_y=("y", "mean")
       )
       .reset_index()
)

# Metrics
cluster_df["AVG_SEVERITY"] = (
    cluster_df["SEVERITY_SUM"] / cluster_df["N_CRASHES"]
)

cluster_df["AVG_CAPPED_SEVERITY"] = (
    cluster_df["CAPPED_SEVERITY_SUM"] / cluster_df["N_CRASHES"]
)

cluster_df["SEVERE_CRASH_PERCENTAGE"] = (
    100 * cluster_df["SEVERE_CRASH_COUNT"] / cluster_df["N_CRASHES"]
)

cluster_df["FATAL_CRASH_PERCENTAGE"] = (
    100 * cluster_df["FATAL_CRASH_COUNT"] / cluster_df["N_CRASHES"]
)

# Convert centroid coords back to lat/lon
centers = gpd.GeoDataFrame(
    cluster_df,
    geometry=gpd.points_from_xy(cluster_df["avg_x"], cluster_df["avg_y"]),
    crs=CRS_METERS
).to_crs(4326)

cluster_df["AVG_LON"] = centers.geometry.x
cluster_df["AVG_LAT"] = centers.geometry.y

# Final clean table
cluster_df = cluster_df.drop(columns=["avg_x", "avg_y"])

cluster_df = cluster_df.sort_values(
    ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

display(cluster_df.head(10))

In [ ]:
import geopandas as gpd

AADT_PATH = "/content/drive/MyDrive/aadt/Traffic_Volume_-_2024.shp"

aadt = gpd.read_file(AADT_PATH)

print(aadt.shape)
print(aadt.columns.tolist())


AADT Normalization by Exposure

In [ ]:
# =====================================================
# AADT MATCHING (FIXED VERSION)
# Uses cluster_df directly since cluster_model_table not created yet
# =====================================================

import geopandas as gpd
import pandas as pd
import numpy as np

CRS_METERS = 32618
YEARS_EXPOSURE = 5
RISK_SCALE = 100_000_000

# -----------------------------------------------------
# 1) Project AADT
# -----------------------------------------------------
aadt_m = aadt.to_crs(CRS_METERS).copy()

aadt_m["AADT"] = pd.to_numeric(aadt_m["AADT"], errors="coerce")
aadt_m = aadt_m[
    aadt_m["AADT"].notna() &
    (aadt_m["AADT"] > 0)
].copy()

print("AADT usable rows:", len(aadt_m))

# -----------------------------------------------------
# 2) Build centroid points from cluster_df
# -----------------------------------------------------
cm = cluster_df.copy().reset_index(drop=True)

cluster_pts = gpd.GeoDataFrame(
    cm,
    geometry=gpd.points_from_xy(
        cm["AVG_LON"],
        cm["AVG_LAT"]
    ),
    crs=4326
).to_crs(CRS_METERS)

# -----------------------------------------------------
# 3) Match nearest AADT segment
# -----------------------------------------------------
match = gpd.sjoin_nearest(
    cluster_pts,
    aadt_m[["AADT", "geometry"]],
    how="left",
    distance_col="DIST_TO_AADT_M"
)

# -----------------------------------------------------
# 4) Compute exposure-normalized risk
# -----------------------------------------------------
match["MATCHED_AADT"] = match["AADT"]

match["EXPOSURE_5YR"] = (
    match["MATCHED_AADT"] * 365 * YEARS_EXPOSURE
)

match["RISK_PER_100M"] = (
    match["SEVERITY_SUM"] / match["EXPOSURE_5YR"]
) * RISK_SCALE

match["CAPPED_RISK_PER_100M"] = (
    match["CAPPED_SEVERITY_SUM"] / match["EXPOSURE_5YR"]
) * RISK_SCALE

# -----------------------------------------------------
# 5) Save back to cluster_df
# -----------------------------------------------------
cluster_df = pd.DataFrame(match.drop(columns="geometry"))

# -----------------------------------------------------
# 6) Diagnostics
# -----------------------------------------------------
print("AADT matching complete.")

display(cluster_df[[
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "EXPOSURE_5YR",
    "RISK_PER_100M"
]].describe())

display(cluster_df[[
    "SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M"
]].head(20))

Table ranked by Normalized by Exposure

In [ ]:
# =====================================================
# RANK BY EXPOSURE-NORMALIZED RISK
# Place AFTER your N_CRASHES filter cell
# =====================================================

# assumes cluster_df already has:
# MATCHED_AADT
# EXPOSURE_5YR
# RISK_PER_100M
# CAPPED_RISK_PER_100M

risk_table = cluster_df.copy()

# apply same minimum crash threshold
risk_table = risk_table[
    risk_table["N_CRASHES"] >= 4
].copy()

# sort by normalized risk
risk_table = risk_table.sort_values(
    ["RISK_PER_100M", "SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

# add rank
risk_table.insert(
    0,
    "RISK_RANK",
    np.arange(1, len(risk_table) + 1)
)

# optional capped rank
risk_table["CAPPED_RISK_RANK"] = (
    risk_table["CAPPED_RISK_PER_100M"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# preview
display(risk_table[[
    "RISK_RANK",
    "N_CRASHES",
    "SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M",
    "CAPPED_RISK_PER_100M"
]].head(30))

print("Rows in risk table:", len(risk_table))

Define model_df as main table

In [ ]:
model_df = cluster_df.copy()

Limit clusters to three crashes minimum

In [ ]:
# =====================================================
# PRIMARY SAMPLE RESTRICTION: N_CRASHES >= 3
# =====================================================

rows_before = len(model_df)

model_df = model_df[
    model_df["N_CRASHES"] >= 4
].copy()

rows_after = len(model_df)

print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("Rows removed:", rows_before - rows_after)
print("Percent retained:", round(100 * rows_after / rows_before, 2), "%")

In [ ]:
# =====================================================
# BUILD PERCENTAGE PREDICTORS FROM CRASH-LEVEL gdf
# Night = 9:00 PM through 5:59 AM local DC time
# =====================================================

tmp = gdf.copy()

# Make sure FROMDATE is parsed as UTC datetime
tmp["FROMDATE"] = pd.to_datetime(tmp["FROMDATE"], errors="coerce", utc=True)

# Convert UTC to Washington, DC local time
tmp["FROMDATE_LOCAL"] = tmp["FROMDATE"].dt.tz_convert("America/New_York")

# --------------------------
# NIGHT FLAG
# --------------------------
# 9 PM through 5:59 AM local time
tmp["IS_NIGHT"] = tmp["FROMDATE_LOCAL"].dt.hour.isin(
    [21, 22, 23, 0, 1, 2, 3, 4, 5]
).astype(int)

# --------------------------
# SPEEDING FLAG
# --------------------------
tmp["IS_SPEEDING"] = (
    pd.to_numeric(tmp["SPEEDING_INVOLVED"], errors="coerce")
      .fillna(0)
      .gt(0)
      .astype(int)
)

# --------------------------
# BIKE FLAG
# --------------------------
tmp["IS_BIKE"] = (
    pd.to_numeric(tmp["TOTAL_BICYCLES"], errors="coerce")
      .fillna(0)
      .gt(0)
      .astype(int)
)

# --------------------------
# AGGREGATE TO CLUSTER LEVEL
# --------------------------
pct_df = (
    tmp.groupby("cluster_id")
       .agg(
           PCT_NIGHT=("IS_NIGHT", "mean"),
           PCT_SPEEDING=("IS_SPEEDING", "mean"),
           PCT_BIKE=("IS_BIKE", "mean")
       )
       .reset_index()
)

pct_df[["PCT_NIGHT", "PCT_SPEEDING", "PCT_BIKE"]] *= 100

# --------------------------
# MERGE INTO MODEL DATAFRAME
# --------------------------
model_df = model_df.copy()


model_df = model_df.merge(
    pct_df,
    on="cluster_id",
    how="left"
)

# --------------------------
# CHECK OUTPUT
# --------------------------
display(model_df[[
    "PCT_NIGHT",
    "PCT_SPEEDING",
    "PCT_BIKE"
]].describe())

display(model_df[[
    "cluster_id",
    "N_CRASHES",
    "SEVERITY_SUM",
    "PCT_NIGHT",
    "PCT_SPEEDING",
    "PCT_BIKE"
]].head(20))

**Creating Nearest Speedcamera/Speedbump infastructure**

import and naming of variables for ddot

In [ ]:
# =====================================================
# SPEED DEVICE DISTANCE — SAME DDOT MATCHING METHOD
# Matches your prior speed notebook methodology
# Adds crash-level: DIST_TO_SPEED_DEVICE_M
# =====================================================

import re
import numpy as np
import pandas as pd
import geopandas as gpd

CAMERAS_PATH       = "/content/drive/MyDrive/Automated_Traffic_Enforcement.csv"
SPEEDHUMPS_PATH    = "/content/drive/MyDrive/Speed_Humps.csv"
SPEED_GEOJSON_PATH = "/content/drive/MyDrive/Roadway_SubBlock.geojson"

CRS_METERS = 32618
MAX_LATERAL_M = 20

# -----------------------------
# Helper: normalize DDOT street names
# -----------------------------
def norm_street(s: str) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).upper().strip()
    s = re.sub(r"\b(NE|NW|SE|SW)\b", "", s)
    s = re.sub(r"[^A-Z0-9 &'-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# -----------------------------
# 1) DDOT lines → street_base
# -----------------------------
gdf_limits_4326 = gpd.read_file(SPEED_GEOJSON_PATH)

if gdf_limits_4326.crs is None:
    gdf_limits_4326 = gdf_limits_4326.set_crs(4326)

gdf_limits_m = gdf_limits_4326.to_crs(CRS_METERS).copy()

name_cands_ddot = [
    c for c in ["STREETNAME", "ROUTENAME", "NAME", "FULLNAME", "SEGMENTNAME"]
    if c in gdf_limits_m.columns
]

ddot_name_col = name_cands_ddot[0] if name_cands_ddot else None

if ddot_name_col:
    gdf_limits_m["street_base"] = gdf_limits_m[ddot_name_col].apply(norm_street)
else:
    gdf_limits_m["street_base"] = ""

ddot_lines = gdf_limits_m[["geometry", "street_base"]].copy()

print("DDOT lines loaded:", len(ddot_lines))
print("DDOT name column used:", ddot_name_col)

More variable refining

In [ ]:
# =====================================================
# 2) LOAD SPEED CAMERAS + SPEED HUMPS
# =====================================================

cams = pd.read_csv(CAMERAS_PATH)

cams = cams.rename(columns={
    "CAMERA_LATITUDE": "LATITUDE",
    "CAMERA_LONGITUDE": "LONGITUDE"
})

cams = cams.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
cams["LATITUDE"] = pd.to_numeric(cams["LATITUDE"], errors="coerce")
cams["LONGITUDE"] = pd.to_numeric(cams["LONGITUDE"], errors="coerce")
cams = cams.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
cams["device_kind"] = "camera"

humps = pd.read_csv(SPEEDHUMPS_PATH)

humps = humps.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
humps["LATITUDE"] = pd.to_numeric(humps["LATITUDE"], errors="coerce")
humps["LONGITUDE"] = pd.to_numeric(humps["LONGITUDE"], errors="coerce")
humps = humps.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
humps["device_kind"] = "hump"

print("Speed cameras:", len(cams))
print("Speed humps:", len(humps))
print("Total speed devices:", len(cams) + len(humps))

Assign Speed Devices to nearest DDOT Line

In [ ]:
# =====================================================
# 3) ASSIGN SPEED DEVICES TO NEAREST DDOT LINE
# Same as prior notebook
# =====================================================

gdf_cams_4326 = gpd.GeoDataFrame(
    cams.copy(),
    geometry=gpd.points_from_xy(cams["LONGITUDE"], cams["LATITUDE"]),
    crs=4326
)

gdf_humps_4326 = gpd.GeoDataFrame(
    humps.copy(),
    geometry=gpd.points_from_xy(humps["LONGITUDE"], humps["LATITUDE"]),
    crs=4326
)

cams_m = gdf_cams_4326.to_crs(CRS_METERS)
humps_m = gdf_humps_4326.to_crs(CRS_METERS)

devices_m = pd.concat(
    [
        cams_m[["geometry", "device_kind"]],
        humps_m[["geometry", "device_kind"]]
    ],
    ignore_index=True
)

dev2line = (
    gpd.sjoin_nearest(
        devices_m,
        ddot_lines,
        how="left",
        distance_col="dev_dist_to_line_m"
    )
    .rename(columns={"index_right": "ddot_idx"})
    .reset_index()
    .rename(columns={"index": "device_index"})
)

dev2line["dev_valid_line"] = dev2line["dev_dist_to_line_m"] <= MAX_LATERAL_M
dev2line["street_base"] = dev2line["street_base"].fillna("").astype(str)

dev_on_street = dev2line[
    dev2line["dev_valid_line"] &
    dev2line["street_base"].ne("")
].copy()

print("Devices valid to street:", len(dev_on_street), "/", len(dev2line))
display(dev_on_street[[
    "device_index",
    "device_kind",
    "street_base",
    "dev_dist_to_line_m"
]].head())

Assign Crashes to nearest DDOT line

In [ ]:
# =====================================================
# 4) ASSIGN BROAD CRASHES TO NEAREST DDOT LINE
# Uses your existing gdf from broad crash notebook
# =====================================================

# gdf should already be projected to CRS_METERS and contain crash_index
# If not, uncomment this:
# gdf = gpd.GeoDataFrame(
#     df.copy(),
#     geometry=gpd.points_from_xy(df["LONGITUDE"], df["LATITUDE"]),
#     crs=4326
# ).to_crs(CRS_METERS)
# gdf = gdf.reset_index(drop=True)
# gdf["crash_index"] = gdf.index

cr2line = (
    gpd.sjoin_nearest(
        gdf,
        ddot_lines,
        how="left",
        distance_col="cr_dist_to_line_m"
    )
    .rename(columns={"index_right": "ddot_idx"})
    .reset_index(drop=True)
)

cr2line["cr_valid_line"] = cr2line["cr_dist_to_line_m"] <= MAX_LATERAL_M
cr2line["street_base"] = cr2line["street_base"].fillna("").astype(str)

cr_on_street = cr2line[
    cr2line["cr_valid_line"] &
    cr2line["street_base"].ne("")
].copy()

print("Crashes valid to street:", len(cr_on_street), "/", len(cr2line))
display(cr_on_street[[
    "crash_index",
    "street_base",
    "cr_dist_to_line_m"
]].head())

Same street distance calculations

In [ ]:
# =====================================================
# 5) SAME-STREET NEAREST SPEED DEVICE DISTANCE
# Same logic as prior speed notebook, but NO filtering
# =====================================================

devices_m_idxed = (
    devices_m.reset_index()
    .rename(columns={"index": "device_index"})
)

devices_m_idxed = devices_m_idxed.merge(
    dev_on_street[["device_index", "street_base"]],
    on="device_index",
    how="left"
)

rows = []

for street, cr_grp in cr_on_street.groupby("street_base"):
    dev_grp = devices_m_idxed[
        devices_m_idxed["street_base"] == street
    ].copy()

    if dev_grp.empty:
        tmp = cr_grp.copy()
        tmp["DIST_TO_SPEED_DEVICE_M"] = np.inf
        tmp["nearest_speed_device_index"] = pd.NA
        tmp["nearest_speed_device_kind"] = pd.NA
        rows.append(tmp)
        continue

    dev_gdf = gpd.GeoDataFrame(
        dev_grp[["geometry", "device_index", "device_kind"]].copy(),
        geometry="geometry",
        crs=CRS_METERS
    )

    tmp = (
        gpd.sjoin_nearest(
            cr_grp.set_geometry("geometry"),
            dev_gdf,
            how="left",
            distance_col="DIST_TO_SPEED_DEVICE_M"
        )
        .rename(columns={"index_right": "_dev_join_row"})
        .copy()
    )

    tmp["nearest_speed_device_index"] = tmp["device_index"]
    tmp["nearest_speed_device_kind"] = tmp["device_kind"]

    rows.append(tmp)

same_street_speed = (
    pd.concat(rows, ignore_index=True)
    if rows
    else cr_on_street.copy()
)

finite = np.isfinite(same_street_speed["DIST_TO_SPEED_DEVICE_M"])

print("Same-street speed-device distances computed.")
print("Finite distances:", int(finite.sum()), "of", len(same_street_speed))
print(same_street_speed["DIST_TO_SPEED_DEVICE_M"].replace(np.inf, np.nan).describe())

same_street_speed = (
    same_street_speed
    .sort_values("DIST_TO_SPEED_DEVICE_M")
    .drop_duplicates(subset=["crash_index"], keep="first")
    .copy()
)

display(same_street_speed[[
    "crash_index",
    "street_base",
    "DIST_TO_SPEED_DEVICE_M",
    "nearest_speed_device_kind"
]].head(10))

Nearest Streetlight Calculations

In [ ]:
# =====================================================
# STREETLIGHT DISTANCE — CRASH LEVEL
# Matches prior night methodology:
# nearest Euclidean distance to streetlight in projected CRS
# Adds separate table: light_dist
# Does NOT merge into gdf yet
# =====================================================

import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.neighbors import BallTree
from pyproj import Transformer

STREETLIGHT_CSV = "/content/drive/MyDrive/Street_Lights.csv"

# --------------------------
# Load streetlights
# --------------------------
streetlights = pd.read_csv(STREETLIGHT_CSV, low_memory=False)

# Normalize columns
streetlights.rename(
    columns={c: c.strip().upper() for c in streetlights.columns},
    inplace=True
)

print("Streetlight columns:")
print(streetlights.columns.tolist())

# --------------------------
# Helpers
# --------------------------
def detect_xy_crs(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")

    xs = x[np.isfinite(x)]
    ys = y[np.isfinite(y)]

    if xs.empty or ys.empty:
        return None, "insufficient finite X/Y values"

    xmin, xmax = xs.min(), xs.max()
    ymin, ymax = ys.min(), ys.max()

    # Already degrees
    if (
        (-180 <= xmin <= 180) and (-180 <= xmax <= 180) and
        (-90 <= ymin <= 90) and (-90 <= ymax <= 90)
    ):
        return "EPSG:4326", "values appear to be degrees"

    # Web Mercator
    if all(abs(v) <= 2.1e7 for v in [xmin, xmax, ymin, ymax]):
        return "EPSG:3857", "values within Web Mercator range"

    # Maryland StatePlane feet
    if all(1e4 <= abs(v) <= 1e7 for v in [xmin, xmax, ymin, ymax]):
        return "EPSG:2248", "values look like StatePlane Maryland feet"

    # Maryland StatePlane meters
    if all(1e3 <= abs(v) <= 1e6 for v in [xmin, xmax, ymin, ymax]):
        return "EPSG:26985", "values look like StatePlane Maryland meters"

    return None, "unable to infer CRS"

# --------------------------
# Coordinate handling
# --------------------------
if {"LATITUDE", "LONGITUDE"}.issubset(streetlights.columns):
    print("Using LATITUDE/LONGITUDE columns.")

elif {"LAT", "LON"}.issubset(streetlights.columns):
    print("Using LAT/LON columns.")
    streetlights = streetlights.rename(
        columns={"LAT": "LATITUDE", "LON": "LONGITUDE"}
    )

elif {"X", "Y"}.issubset(streetlights.columns):
    src_epsg, reason = detect_xy_crs(streetlights["X"], streetlights["Y"])

    if src_epsg is None:
        raise ValueError(f"Could not infer X/Y CRS: {reason}")

    print(f"Converting X/Y to WGS84 from {src_epsg}: {reason}")

    tf = Transformer.from_crs(src_epsg, "EPSG:4326", always_xy=True)

    lon, lat = tf.transform(
        pd.to_numeric(streetlights["X"], errors="coerce").to_numpy(),
        pd.to_numeric(streetlights["Y"], errors="coerce").to_numpy()
    )

    streetlights["LATITUDE"] = lat
    streetlights["LONGITUDE"] = lon

else:
    raise ValueError("Could not find LAT/LON, LATITUDE/LONGITUDE, or X/Y columns.")

# --------------------------
# Clean coordinates
# --------------------------
streetlights = streetlights.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

streetlights["LATITUDE"] = pd.to_numeric(
    streetlights["LATITUDE"], errors="coerce"
)

streetlights["LONGITUDE"] = pd.to_numeric(
    streetlights["LONGITUDE"], errors="coerce"
)

streetlights = streetlights.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

# Fix common DC longitude issue
if streetlights["LONGITUDE"].median() > 0:
    print("Positive longitude detected; flipping sign.")
    streetlights["LONGITUDE"] = -streetlights["LONGITUDE"].abs()

# Remove exact duplicates
streetlights = streetlights.drop_duplicates(
    subset=["LATITUDE", "LONGITUDE"]
).reset_index(drop=True)

# Optional: remove removed/decommissioned assets
if "ASSETSTATUS" in streetlights.columns:
    before = len(streetlights)
    streetlights = streetlights[
        ~streetlights["ASSETSTATUS"]
        .astype(str)
        .str.contains("Removed|Decommissioned", case=False, na=False)
    ].copy()
    print("Removed/decommissioned streetlights dropped:", before - len(streetlights))

# Clip to plausible DC area
LAT_MIN, LAT_MAX = 38.81, 38.995
LON_MIN, LON_MAX = -77.12, -76.91

before = len(streetlights)

streetlights = streetlights[
    streetlights["LATITUDE"].between(LAT_MIN, LAT_MAX) &
    streetlights["LONGITUDE"].between(LON_MIN, LON_MAX)
].copy()

print(f"Streetlights kept in DC bounds: {len(streetlights):,} / {before:,}")

if streetlights.empty:
    raise ValueError("Streetlight dataset empty after cleaning.")

# --------------------------
# Project streetlights to meters
# --------------------------
lights_gdf = gpd.GeoDataFrame(
    streetlights.copy(),
    geometry=gpd.points_from_xy(
        streetlights["LONGITUDE"],
        streetlights["LATITUDE"]
    ),
    crs=4326
).to_crs(CRS_METERS)

# --------------------------
# Nearest streetlight distance
# gdf must already be in CRS_METERS and have crash_index
# --------------------------
tree = BallTree(
    np.c_[lights_gdf.geometry.x.values, lights_gdf.geometry.y.values],
    metric="euclidean"
)

dist_m, nearest_idx = tree.query(
    np.c_[gdf.geometry.x.values, gdf.geometry.y.values],
    k=1
)

light_dist = pd.DataFrame({
    "crash_index": gdf["crash_index"].values,
    "DIST_TO_LIGHT_M": dist_m.flatten(),
    "nearest_light_index": nearest_idx.flatten()
})

# --------------------------
# Diagnostics
# --------------------------
print("\nStreetlight distance complete.")
print("Rows:", len(light_dist))
print(light_dist["DIST_TO_LIGHT_M"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

display(light_dist.head(10))

Nearest Bike Lane Calculations

In [ ]:
# =====================================================
# FASTER BIKE LANE DISTANCE — CRASH LEVEL
# Direct Euclidean nearest bike lane distance
# No DDOT matching, no thresholding
# =====================================================

import geopandas as gpd
import pandas as pd
import numpy as np

BIKELANES_PATH = "/content/drive/MyDrive/Bicycle_Lanes.geojson"

lanes = gpd.read_file(BIKELANES_PATH)

if lanes.crs is None:
    lanes = lanes.set_crs(4326)

lanes = lanes[
    lanes.geometry.notna() &
    ~lanes.geometry.is_empty &
    lanes.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()

lanes_m = lanes.to_crs(CRS_METERS).copy()

print("Bike lane features:", len(lanes_m))
print("Crash rows:", len(gdf))

# keep only geometry to avoid duplicate columns
lanes_min = lanes_m[["geometry"]].copy()

bike_join = gpd.sjoin_nearest(
    gdf[["crash_index", "geometry"]],
    lanes_min,
    how="left",
    distance_col="DIST_TO_BIKE_LANE_M"
)

# If ties create duplicates, keep closest row per crash
bike_dist = (
    bike_join[["crash_index", "DIST_TO_BIKE_LANE_M"]]
    .sort_values("DIST_TO_BIKE_LANE_M")
    .drop_duplicates("crash_index", keep="first")
    .copy()
)

print("\nBike distance complete.")
print("bike_dist rows:", len(bike_dist))
print("unique crashes:", bike_dist["crash_index"].nunique())
print("missing distances:", bike_dist["DIST_TO_BIKE_LANE_M"].isna().sum())

print(bike_dist["DIST_TO_BIKE_LANE_M"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
))

display(bike_dist.head(10))

Add in the Median Nearest Distance to Infastructure and Create final table

In [ ]:
# =====================================================
# FINAL CLUSTER-LEVEL TABLE:
# Rank, crashes, severity, percentages, interaction percentages,
# and median infrastructure distances
# =====================================================

import numpy as np
import pandas as pd

# --------------------------
# 1) Build crash-level flags from gdf
# --------------------------
tmp = gdf.copy()

# Date → local time
tmp["FROMDATE"] = pd.to_datetime(tmp["FROMDATE"], errors="coerce", utc=True)
tmp["FROMDATE_LOCAL"] = tmp["FROMDATE"].dt.tz_convert("America/New_York")

# Night = 9 PM through 5:59 AM
tmp["IS_NIGHT"] = tmp["FROMDATE_LOCAL"].dt.hour.isin(
    [21, 22, 23, 0, 1, 2, 3, 4, 5]
).astype(int)

# Speeding
tmp["IS_SPEEDING"] = (
    pd.to_numeric(tmp["SPEEDING_INVOLVED"], errors="coerce")
      .fillna(0)
      .gt(0)
      .astype(int)
)

# Bike
tmp["IS_BIKE"] = (
    pd.to_numeric(tmp["TOTAL_BICYCLES"], errors="coerce")
      .fillna(0)
      .gt(0)
      .astype(int)
)

# Interaction flags
tmp["IS_NIGHT_AND_BIKE"] = (
    (tmp["IS_NIGHT"] == 1) &
    (tmp["IS_BIKE"] == 1)
).astype(int)

tmp["IS_NIGHT_AND_SPEED"] = (
    (tmp["IS_NIGHT"] == 1) &
    (tmp["IS_SPEEDING"] == 1)
).astype(int)

tmp["IS_SPEED_AND_BIKE"] = (
    (tmp["IS_SPEEDING"] == 1) &
    (tmp["IS_BIKE"] == 1)
).astype(int)

# --------------------------
# 2) Cluster-level percentage predictors
# --------------------------
cluster_pct = (
    tmp.groupby("cluster_id", as_index=False)
       .agg(
           PCT_NIGHT=("IS_NIGHT", "mean"),
           PCT_SPEEDING=("IS_SPEEDING", "mean"),
           PCT_BIKE=("IS_BIKE", "mean"),
           PCT_NIGHT_AND_BIKE=("IS_NIGHT_AND_BIKE", "mean"),
           PCT_NIGHT_AND_SPEED=("IS_NIGHT_AND_SPEED", "mean"),
           PCT_SPEED_AND_BIKE=("IS_SPEED_AND_BIKE", "mean")
       )
)

pct_cols = [
    "PCT_NIGHT",
    "PCT_SPEEDING",
    "PCT_BIKE",
    "PCT_NIGHT_AND_BIKE",
    "PCT_NIGHT_AND_SPEED",
    "PCT_SPEED_AND_BIKE"
]

cluster_pct[pct_cols] = cluster_pct[pct_cols] * 100

# --------------------------
# 3) Clean speed distance table
# --------------------------
speed_dist = (
    same_street_speed[["crash_index", "DIST_TO_SPEED_DEVICE_M"]]
    .sort_values("DIST_TO_SPEED_DEVICE_M")
    .drop_duplicates("crash_index", keep="first")
    .copy()
)

# --------------------------
# 4) Build crash-level infrastructure table
# --------------------------
infra_crash = (
    gdf[["crash_index", "cluster_id"]]
    .drop_duplicates("crash_index")
    .merge(speed_dist, on="crash_index", how="left")
    .merge(
        light_dist[["crash_index", "DIST_TO_LIGHT_M"]],
        on="crash_index",
        how="left"
    )
    .merge(
        bike_dist[["crash_index", "DIST_TO_BIKE_LANE_M"]],
        on="crash_index",
        how="left"
    )
)

# Speed: no same-street device / unmatched = very far
infra_crash["DIST_TO_SPEED_DEVICE_M"] = (
    infra_crash["DIST_TO_SPEED_DEVICE_M"]
    .replace(np.inf, np.nan)
    .clip(upper=5000)
    .fillna(5000)
)

# --------------------------
# 5) Cluster-level median distances
# --------------------------
cluster_infra = (
    infra_crash.groupby("cluster_id", as_index=False)
       .agg(
           MEDIAN_DIST_SPEED_DEVICE_M=("DIST_TO_SPEED_DEVICE_M", "median"),
           MEDIAN_DIST_LIGHT_M=("DIST_TO_LIGHT_M", "median"),
           MEDIAN_DIST_BIKE_LANE_M=("DIST_TO_BIKE_LANE_M", "median")
       )
)

# --------------------------
# 6) Start from cluster_df, add percentages + distances
# --------------------------
add_cols = pct_cols + [
    "MEDIAN_DIST_SPEED_DEVICE_M",
    "MEDIAN_DIST_LIGHT_M",
    "MEDIAN_DIST_BIKE_LANE_M"
]

cluster_model_table = cluster_df.drop(
    columns=[c for c in add_cols if c in cluster_df.columns],
    errors="ignore"
).copy()

cluster_model_table = (
    cluster_model_table
    .merge(cluster_pct, on="cluster_id", how="left")
    .merge(cluster_infra, on="cluster_id", how="left")
)

# --------------------------
# 7) Apply model restriction if desired: N_CRASHES >= 4
# --------------------------
cluster_model_table = cluster_model_table[
    cluster_model_table["N_CRASHES"] >= 4
].copy()

# --------------------------
# 8) Rank by severity
# --------------------------
cluster_model_table = cluster_model_table.sort_values(
    ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

cluster_model_table.insert(
    0,
    "RANK",
    np.arange(1, len(cluster_model_table) + 1)
)

# --------------------------
# 9) Final display columns
# --------------------------
final_cols = [
    "RANK",
    "N_CRASHES",
    "SEVERITY_SUM",
    "AVG_SEVERITY",
    "SEVERE_CRASH_PERCENTAGE",
    "FATAL_CRASH_PERCENTAGE",
    "PCT_NIGHT",
    "PCT_SPEEDING",
    "PCT_BIKE",
    "PCT_NIGHT_AND_BIKE",
    "PCT_NIGHT_AND_SPEED",
    "PCT_SPEED_AND_BIKE",
    "MEDIAN_DIST_SPEED_DEVICE_M",
    "MEDIAN_DIST_LIGHT_M",
    "MEDIAN_DIST_BIKE_LANE_M"
]

cluster_model_table = cluster_model_table[final_cols].copy()

# Optional: cleaner rounding
round_cols = [c for c in cluster_model_table.columns if c != "RANK" and c != "N_CRASHES"]
cluster_model_table[round_cols] = cluster_model_table[round_cols].round(2)

display(cluster_model_table.head(30))

print("Final table shape:", cluster_model_table.shape)
print("\nMissing values:")
display(cluster_model_table.isna().sum())


Create Three Logistical Regressions, one with a focus on top quartile severity sum, one with a focus on quartile avg severity, one with a focus on top quartile severe crash percentage.

In [ ]:
# =====================================================
# THREE LOGISTIC REGRESSIONS — CLEAN STATSMODELS OUTPUT
# Same visual as before
# =====================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

base_reg = cluster_model_table.copy()

X_cols = [
    "PCT_NIGHT",
    "PCT_SPEEDING",
    "PCT_BIKE",
    "PCT_NIGHT_AND_BIKE",
    "PCT_NIGHT_AND_SPEED",
    "PCT_SPEED_AND_BIKE",
    "MEDIAN_DIST_SPEED_DEVICE_M",
    "MEDIAN_DIST_LIGHT_M",
    "MEDIAN_DIST_BIKE_LANE_M"
]

def run_logit_summary(df, target_col, outcome_name):
    df_reg = df.copy()

    cutoff = df_reg[target_col].quantile(0.75)
    df_reg[outcome_name] = (df_reg[target_col] >= cutoff).astype(int)

    print("\n" + "="*90)
    print(f"MODEL: {outcome_name}")
    print(f"Target column: {target_col}")
    print(f"75th percentile cutoff: {cutoff}")
    print(df_reg[outcome_name].value_counts())
    print("="*90)

    df_reg = df_reg.dropna(subset=X_cols + [outcome_name]).copy()

    X = df_reg[X_cols].copy()
    y = df_reg[outcome_name]

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns,
        index=X.index
    )

    X_scaled = sm.add_constant(X_scaled)

    model = sm.Logit(y, X_scaled)
    result = model.fit(maxiter=200)

    print(result.summary())

    odds_ratios = pd.DataFrame({
        "Variable": result.params.index,
        "Coef": result.params.values,
        "Odds_Ratio": np.exp(result.params.values),
        "P_Value": result.pvalues.values
    })

    display(odds_ratios.sort_values("P_Value"))

    return result, odds_ratios


severity_sum_result, severity_sum_odds = run_logit_summary(
    base_reg,
    target_col="SEVERITY_SUM",
    outcome_name="HIGH_TOTAL_SEVERITY"
)

avg_severity_result, avg_severity_odds = run_logit_summary(
    base_reg,
    target_col="AVG_SEVERITY",
    outcome_name="HIGH_AVG_SEVERITY"
)

severe_pct_result, severe_pct_odds = run_logit_summary(
    base_reg,
    target_col="SEVERE_CRASH_PERCENTAGE",
    outcome_name="HIGH_SEVERE_CRASH_PERCENTAGE"
)

Rebuild regression table based on normalization by exposure

In [ ]:
# =====================================================
# BUILD RISK REGRESSION TABLE
# cluster_df has risk metrics
# cluster_pct has percentage predictors
# cluster_infra has distance predictors
# =====================================================

risk_reg = (
    cluster_df
    .merge(cluster_pct, on="cluster_id", how="left")
    .merge(cluster_infra, on="cluster_id", how="left")
    .copy()
)

# Same sample restriction
risk_reg = risk_reg[risk_reg["N_CRASHES"] >= 4].copy()

# Rank by exposure-normalized risk
risk_reg = risk_reg.sort_values(
    ["RISK_PER_100M", "SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

risk_reg.insert(0, "RISK_RANK", np.arange(1, len(risk_reg) + 1))



Logistic regression based on normalization by exposure

In [ ]:
# =====================================================
# LOGISTIC REGRESSION — HIGH EXPOSURE-NORMALIZED RISK
# Outcome: top quartile of RISK_PER_100M
# =====================================================

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from IPython.display import display

predictor_cols = [
    "PCT_NIGHT",
    "PCT_SPEEDING",
    "PCT_BIKE",
    "PCT_NIGHT_AND_BIKE",
    "PCT_NIGHT_AND_SPEED",
    "PCT_SPEED_AND_BIKE",
    "MEDIAN_DIST_SPEED_DEVICE_M",
    "MEDIAN_DIST_LIGHT_M",
    "MEDIAN_DIST_BIKE_LANE_M"
]

cutoff = risk_reg["RISK_PER_100M"].quantile(0.75)

risk_reg["HIGH_EXPOSURE_NORMALIZED_RISK"] = (
    risk_reg["RISK_PER_100M"] >= cutoff
).astype(int)

print("75th percentile RISK_PER_100M cutoff:", cutoff)
print(risk_reg["HIGH_EXPOSURE_NORMALIZED_RISK"].value_counts())

df_reg = risk_reg.dropna(
    subset=predictor_cols + ["HIGH_EXPOSURE_NORMALIZED_RISK"]
).copy()

X = df_reg[predictor_cols]
y = df_reg["HIGH_EXPOSURE_NORMALIZED_RISK"]

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

X_scaled = sm.add_constant(X_scaled)

model = sm.Logit(y, X_scaled)
risk_result = model.fit(maxiter=200)

print(risk_result.summary())

risk_odds = pd.DataFrame({
    "Variable": risk_result.params.index,
    "Coef": risk_result.params.values,
    "Odds_Ratio": np.exp(risk_result.params.values),
    "P_Value": risk_result.pvalues.values
}).sort_values("P_Value")

display(risk_odds)

Table for Publication

In [ ]:
# =====================================================
# TABLE 2 — LOGISTIC REGRESSION RESULTS ACROSS OUTCOMES
# Each cell reports: odds ratio, p-value
# =====================================================

import pandas as pd
import numpy as np
from IPython.display import display

# -----------------------------
# 1) Clean predictor names
# -----------------------------
predictor_name_map = {
    "PCT_NIGHT": "Night crash percentage",
    "PCT_SPEEDING": "Speeding crash percentage",
    "PCT_BIKE": "Bike-involved crash percentage",
    "PCT_NIGHT_AND_BIKE": "Night and bike-involved crash percentage",
    "PCT_NIGHT_AND_SPEED": "Night and speeding crash percentage",
    "PCT_SPEED_AND_BIKE": "Speeding and bike-involved crash percentage",
    "MEDIAN_DIST_SPEED_DEVICE_M": "Median distance to nearest speed-related device",
    "MEDIAN_DIST_LIGHT_M": "Median distance to nearest streetlight",
    "MEDIAN_DIST_BIKE_LANE_M": "Median distance to nearest bike lane"
}

predictor_order = [
    "PCT_NIGHT",
    "PCT_SPEEDING",
    "PCT_BIKE",
    "PCT_NIGHT_AND_SPEED",
    "PCT_NIGHT_AND_BIKE",
    "PCT_SPEED_AND_BIKE",
    "MEDIAN_DIST_LIGHT_M",
    "MEDIAN_DIST_SPEED_DEVICE_M",
    "MEDIAN_DIST_BIKE_LANE_M"
]

# -----------------------------
# 2) Helper function to format each regression result
# -----------------------------
def format_odds_table(odds_df, model_col_name):
    out = odds_df.copy()

    # Remove intercept/constant row
    out = out[out["Variable"] != "const"].copy()

    # Keep only needed columns
    out = out[["Variable", "Odds_Ratio", "P_Value"]].copy()

    # Format cell as: odds ratio, p = value
    out[model_col_name] = out.apply(
        lambda row: f"{row['Odds_Ratio']:.2f}, p = {row['P_Value']:.3f}",
        axis=1
    )

    return out[["Variable", model_col_name]]

# -----------------------------
# 3) Format each model
# -----------------------------
severity_sum_table = format_odds_table(
    severity_sum_odds,
    "Severity Sum"
)

avg_severity_table = format_odds_table(
    avg_severity_odds,
    "Average Severity"
)

severe_percentage_table = format_odds_table(
    severe_pct_odds,
    "Serious-Injury Percentage"
)

exposure_normalized_table = format_odds_table(
    risk_odds,
    "Exposure-Normalized Severity"
)

# -----------------------------
# 4) Merge into one final table
# -----------------------------
table_2 = (
    severity_sum_table
    .merge(avg_severity_table, on="Variable", how="outer")
    .merge(severe_percentage_table, on="Variable", how="outer")
    .merge(exposure_normalized_table, on="Variable", how="outer")
)

# -----------------------------
# 5) Apply clean labels and ordering
# -----------------------------
table_2["Predictor"] = table_2["Variable"].map(predictor_name_map)

table_2["Predictor_Order"] = table_2["Variable"].map(
    {var: i for i, var in enumerate(predictor_order)}
)

table_2 = (
    table_2
    .sort_values("Predictor_Order")
    .drop(columns=["Variable", "Predictor_Order"])
)

# Put Predictor first
table_2 = table_2[
    [
        "Predictor",
        "Severity Sum",
        "Average Severity",
        "Serious-Injury Percentage",
        "Exposure-Normalized Severity"
    ]
]
table_2 = table_2.reset_index(drop=True)
display(table_2.style.hide(axis="index"))

# -----------------------------
# 6) Display final table
# -----------------------------

# Optional: save to CSV
table_2.to_csv("/content/drive/MyDrive/logistic_regression_table_2.csv", index=False)

print("Saved Table 2 to: /content/drive/MyDrive/logistic_regression_table_2.csv")